In [1]:
import os
import pickle
from tqdm import tqdm
from glob import glob

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, load_metric

import torch
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments,
                          DataCollatorForSeq2Seq, Trainer, pipeline)
from peft import (LoraConfig, get_peft_model, TaskType,
                  PeftModel, PeftConfig)

import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level = logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

In [2]:
class CFG:
    wandb = True
    report_to = None
    lab_assignment = 3
    _wandb_kernel = "temuujin"

    debug = False
    num_workers = 12

    prefix_val = "summarize: "
    output_dir = "processed_data"
    model_save_dir = "PEFT_T5"

    tokenizer_name = "google/t5-efficient-mini"
    model_name = "google/t5-efficient-mini"

    project = 'NUM-Machine-Learning-Lab-3'
    name = "Lab 3 Model Training - Text Summarization T5, 90 epochs"

    config = {
        "output_dir": "t5_small_lab3_finetune_PEFT",
        "group": model_name,
        "learning_rate": 2e-5,
        "weight_decay": 1e-3,
        'num_train_epochs': 90,
        "train_batch_size": 13,
        "eval_batch_size": 8,
        "max_seq_length": 1024,
        "overwrite_output_dir": True,
        "reprocess_input_data": True,
        "fp16": True
    }

    test_size = 0.2

    train = True
    eval = True

    eval_metric = "rouge"

    early_stopping_patience = 15

if CFG.debug:
    CFG.config['num_train_epochs'] = 2

if CFG.wandb:
    os.environ["WANDB_SILENT"] = "True"
    CFG.report_to = "wandb"

    import wandb
    wandb.login()

    run = wandb.init(
        project = CFG.project,
        name = CFG.name,
        config = CFG.config
    )

config = CFG.config

# 1. Өгөгдлөө модел сургахад бэлтгэх

In [3]:
df_names = glob("../Lab 2/processed_dfs/*.parquet")
search_terms = '\n'.join([os.path.basename(filename).split('_')[1] for filename in df_names])

df = pd.DataFrame()
for df_name in tqdm(df_names):
    df = pd.concat([df, pd.read_parquet(df_name)[['title', 'abstract']]])

df.drop_duplicates(inplace = True)
df.reset_index(drop = True, inplace = True)

df.rename(columns = {'title': 'target_text', 'abstract': 'input_text'}, inplace = True)
df['prefix'] = CFG.prefix_val

os.makedirs(CFG.output_dir, exist_ok = True)
output_filename = os.path.join(CFG.output_dir, "arxiv_title_generation.parquet")
df.to_parquet(output_filename)

100%|██████████| 12/12 [00:00<00:00, 17.60it/s]


In [4]:
print("\nDataframe memory usage")
print(df.memory_usage(deep = True))

print(f"Dataframe shape: {df.shape}\n")
print(f"All search terms:\n{search_terms}")

print(df.head())


Dataframe memory usage
Index               132
target_text     1966107
input_text     17616445
prefix           983416
dtype: int64
Dataframe shape: (14462, 3)

All search terms:
audio+classification
audio+deep+learning
audio+encoding
audio+fast+fourier
audio+fourier
audio+generation
audio+machine+learning
audio+prediction
audio+recognition
audio+representation
audio+restoration
audio+signal
                                         target_text  \
0  Improved Mispronunciation detection system usi...   
1  Improving Factored Hybrid HMM Acoustic Modelin...   
2  Disentangling Style and Speaker Attributes for...   
3  Synthetic speech detection using meta-learning...   
4  A Pre-trained Audio-Visual Transformer for Emo...   

                                          input_text       prefix  
0  This report proposes state-of-the-art research...  summarize:   
1  In this work, we show that a factored hybrid h...  summarize:   
2  End-to-end neural TTS has shown improved perfo...  summarize

In [5]:
test_size = CFG.test_size
test_df = df.sample(frac = test_size, random_state = 1970)
train_df = df.drop(index = test_df.index)

print(f"Training instance count: {len(train_df)}\nTest instance count: {len(test_df)}\n")

train_dataset = Dataset.from_dict(train_df)
test_dataset = Dataset.from_dict(test_df)
arxiv_title_dict = DatasetDict({"train": train_dataset,"test": test_dataset})

output_filename = os.path.join(CFG.output_dir, "arxiv_title_generation_dataset")
arxiv_title_dict.save_to_disk(output_filename)

print(arxiv_title_dict)

Training instance count: 11570
Test instance count: 2892



Saving the dataset (0/1 shards):   0%|          | 0/11570 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2892 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['target_text', 'input_text', 'prefix'],
        num_rows: 11570
    })
    test: Dataset({
        features: ['target_text', 'input_text', 'prefix'],
        num_rows: 2892
    })
})


In [6]:
tokenizer = AutoTokenizer.from_pretrained(CFG.tokenizer_name, use_fast = False)

def preprocess_function(examples):
    inputs = [CFG.prefix_val + doc for doc in examples["input_text"]]
    model_inputs = tokenizer(inputs,
                             max_length = config['max_seq_length'],
                             padding = "max_length",
                             truncation = True,
                             return_tensors = 'pt')

    labels = tokenizer(text_target = examples["target_text"],
                       max_length = config["max_seq_length"] // 4,
                       padding = "max_length",
                       truncation = True,
                       return_tensors = 'pt')
    
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_arxiv = arxiv_title_dict.map(preprocess_function, batched = True)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Map:   0%|          | 0/11570 [00:00<?, ? examples/s]

Map:   0%|          | 0/2892 [00:00<?, ? examples/s]

# Parameter Efficient Fine-Tuning (PEFT)

## 2.1 Model Fine-Tuning

In [7]:
lora_config = LoraConfig(
    task_type = TaskType.SEQ_2_SEQ_LM, 
    inference_mode = False, 
    r = 8, 
    lora_alpha = 32, 
    lora_dropout = 0.3
)

model = AutoModelForSeq2SeqLM.from_pretrained(CFG.model_name)
data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer, model = CFG.model_name)

peft_model = get_peft_model(model, lora_config)

peft_model.print_trainable_parameters()

trainable params: 172,032 || all params: 31,392,512 || trainable%: 0.548003294543616


In [8]:
training_args = TrainingArguments(
    report_to = CFG.report_to,
    output_dir = config["output_dir"],
    num_train_epochs = config["num_train_epochs"],
    per_device_train_batch_size = config["train_batch_size"],
    overwrite_output_dir = config["overwrite_output_dir"],
    learning_rate = config["learning_rate"],
    weight_decay = config["weight_decay"],
    do_eval = False,
    disable_tqdm = False,
)

trainer = Trainer(
    model = peft_model,
    args = training_args,
    train_dataset = tokenized_arxiv["train"],
    data_collator = data_collator
)

In [9]:
torch.cuda.empty_cache()

trainer.train()

  0%|          | 0/80100 [00:00<?, ?it/s]

{'loss': 15.4333, 'grad_norm': 5.308463096618652, 'learning_rate': 1.9875156054931336e-05, 'epoch': 0.56}
{'loss': 4.9357, 'grad_norm': 0.8433196544647217, 'learning_rate': 1.9750312109862673e-05, 'epoch': 1.12}
{'loss': 0.7226, 'grad_norm': 0.37632179260253906, 'learning_rate': 1.9625468164794007e-05, 'epoch': 1.69}
{'loss': 0.6039, 'grad_norm': 0.3556130826473236, 'learning_rate': 1.9500624219725347e-05, 'epoch': 2.25}
{'loss': 0.53, 'grad_norm': 0.38089191913604736, 'learning_rate': 1.937578027465668e-05, 'epoch': 2.81}
{'loss': 0.4821, 'grad_norm': 0.3068518340587616, 'learning_rate': 1.925093632958802e-05, 'epoch': 3.37}
{'loss': 0.4495, 'grad_norm': 0.325992226600647, 'learning_rate': 1.9126092384519352e-05, 'epoch': 3.93}
{'loss': 0.4255, 'grad_norm': 0.2628606855869293, 'learning_rate': 1.900124843945069e-05, 'epoch': 4.49}
{'loss': 0.4117, 'grad_norm': 0.2989678680896759, 'learning_rate': 1.8876404494382024e-05, 'epoch': 5.06}
{'loss': 0.3925, 'grad_norm': 0.2576594352722168, 

TrainOutput(global_step=80100, training_loss=0.41918015521712665, metrics={'train_runtime': 17629.1702, 'train_samples_per_second': 59.067, 'train_steps_per_second': 4.544, 'train_loss': 0.41918015521712665, 'epoch': 90.0})

In [10]:
test_ver = 8
model_save_path = os.path.join(CFG.model_save_dir, f"test_v{test_ver}")
os.makedirs(model_save_path, exist_ok = True)

trainer.model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"Saved model to: {model_save_path}")

Saved model to: PEFT_T5/test_v8


## 2.2 Evaluation

In [11]:
rouge = load_metric(CFG.eval_metric)

def compute_metrics(decoded_preds, decoded_labels):
    rouge_scores = rouge.compute(predictions = decoded_preds, 
                                 references = decoded_labels, 
                                 rouge_types = ["rouge1"])["rouge1"]
    return rouge_scores

def print_rouge_scores(model_results):
    rouge1_low = model_results.low
    rouge1_mid = model_results.mid
    rouge1_high = model_results.high

    print("| ROUGE Type     | Precision | Recall    | F1-Score  |")
    print("-" * 54)

    print("| ROUGE-1 (Low ) | {:.4f}    | {:.4f}    | {:.4f}    |".format(rouge1_low.precision, rouge1_low.recall, rouge1_low.fmeasure))
    print("| ROUGE-1 (Mid ) | {:.4f}    | {:.4f}    | {:.4f}    |".format(rouge1_mid.precision, rouge1_mid.recall, rouge1_mid.fmeasure))
    print("| ROUGE-1 (High) | {:.4f}    | {:.4f}    | {:.4f}    |".format(rouge1_high.precision, rouge1_high.recall, rouge1_high.fmeasure))

In [12]:
#peft_model_id = "./PEFT_T5/test_v8"
peft_model_id = model_save_path
config = PeftConfig.from_pretrained(peft_model_id)

model = AutoModelForSeq2SeqLM.from_pretrained(config.base_model_name_or_path)
model = PeftModel.from_pretrained(model, peft_model_id)

tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

In [13]:
torch.cuda.empty_cache()

device = 'cuda'

model = model.to(device)
model.eval()

batch_size = 16

with torch.no_grad():
    all_decoded_results = []

    for i in tqdm(range(0, len(tokenized_arxiv['test']['input_ids']), batch_size), 
                  total = len(tokenized_arxiv['test']['input_ids']) // batch_size + 1):
        batch_input_ids = torch.tensor(tokenized_arxiv['test']['input_ids'][i:i+batch_size]).to(device)
        
        batch_encoded_results = model.generate(input_ids = batch_input_ids, 
                                               max_new_tokens = 10)
        batch_decoded_results = tokenizer.batch_decode(batch_encoded_results.detach().cpu().numpy(), 
                                                       skip_special_tokens = True)
        all_decoded_results.extend(batch_decoded_results)

file_path = "./PEFT_T5/test_v7/all_decoded_results.pkl"
with open(file_path, "wb") as file:
    pickle.dump(all_decoded_results, file)
print(f"Decoded predictions saved to: {file_path}")

100%|██████████| 181/181 [03:58<00:00,  1.32s/it]

Decoded predictions saved to: ./PEFT_T5/test_v7/all_decoded_results.pkl


In [14]:
model_results = compute_metrics(all_decoded_results, tokenized_arxiv['test']['target_text'])
print_rouge_scores(model_results)

INFO:absl:Using default tokenizer.


| ROUGE Type     | Precision | Recall    | F1-Score  |
------------------------------------------------------
| ROUGE-1 (Low ) | 0.4103    | 0.2345    | 0.2883    |
| ROUGE-1 (Mid ) | 0.4205    | 0.2409    | 0.2955    |
| ROUGE-1 (High) | 0.4310    | 0.2473    | 0.3029    |


## 2.3 Checking out some predictions

In [15]:
def print_random_samples(all_decoded_results, target_texts, num_samples = 9):
  rand_indices = np.random.randint(1, tokenized_arxiv['test'].num_rows, size = num_samples)
  rand_pred_title = np.asarray(all_decoded_results)[rand_indices]
  rand_target_title = np.asarray(target_texts)[rand_indices]

  for idx, (predicted_title, actual_title) in enumerate(zip(rand_pred_title, rand_target_title)):
      print(f"Test index: {rand_indices[idx]}")
      print("Predicted Title:", predicted_title)
      print("Actual Title:", actual_title)
      print("-" * 64)

print_random_samples(all_decoded_results, tokenized_arxiv['test']['target_text'], num_samples = 3)

Test index: 861
Predicted Title: VS-based Speech Generation for Speech Generation
Actual Title: Evince the artifacts of Spoof Speech by blending Vocal Tract and Voice Source Features
----------------------------------------------------------------
Test index: 1295
Predicted Title: Automatic Recognition and Diagnostic Tool for Screening respiratory infections
Actual Title: Can Machine Learning Be Used to Recognize and Diagnose Coughs?
----------------------------------------------------------------
Test index: 1131
Predicted Title: Deep Neural Network: Deep Neural Network and
Actual Title: A Comparison of deep learning methods for environmental sound
----------------------------------------------------------------


# Full Model Fine-Tuning: T5 small

## 3.1 Loading already finetuned model

In [16]:
full_finetuned_model = pipeline("summarization", model = "./t5_small_lab3_finetune/checkpoint-11500", batch_size = batch_size)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


## 3.2 Full model evaluation

In [17]:
torch.cuda.empty_cache()

with torch.no_grad():
    all_decoded_results = []

    for i in tqdm(range(0, len(tokenized_arxiv['test']['input_ids']), batch_size), 
                  total = len(tokenized_arxiv['test']['input_ids']) // batch_size + 1):
        batch_input = [CFG.prefix_val + input_text for input_text in tokenized_arxiv['test']['input_text'][i:i+batch_size]]
        pred = full_finetuned_model(batch_input)
        pred = [pred_text['summary_text'] for pred_text in pred]
        
        all_decoded_results.extend(pred)

file_path = "./PEFT_T5/test_v7/full_model_finetune_test_all_decoded_results.pkl"
with open(file_path, "wb") as file:
    pickle.dump(all_decoded_results, file)
print(f"Decoded predictions saved to: {file_path}")

100%|██████████| 181/181 [04:49<00:00,  1.60s/it]

Decoded predictions saved to: ./PEFT_T5/test_v7/full_model_finetune_test_all_decoded_results.pkl


In [18]:
model_results = compute_metrics(all_decoded_results, tokenized_arxiv['test']['target_text'])
print_rouge_scores(model_results)

INFO:absl:Using default tokenizer.


| ROUGE Type     | Precision | Recall    | F1-Score  |
------------------------------------------------------
| ROUGE-1 (Low ) | 0.3963    | 0.3066    | 0.3338    |
| ROUGE-1 (Mid ) | 0.4056    | 0.3148    | 0.3421    |
| ROUGE-1 (High) | 0.4146    | 0.3225    | 0.3492    |


In [19]:
print_random_samples(all_decoded_results, tokenized_arxiv['test']['target_text'], num_samples = 3)

Test index: 1096
Predicted Title: Using Unlabeled Data for Unlabeled Video Data
Actual Title: Evolving Losses for Unlabeled Video Representation Learning
----------------------------------------------------------------
Test index: 1639
Predicted Title: a temporal-spatial neural filter for Target Speech Separation
Actual Title: Temporal-Spatial Neural Filter: Direction Informed End-to-End Multi-channel Target Speech Separation
----------------------------------------------------------------
Test index: 2170
Predicted Title: XXX: A novel system for Speech Recognition
Actual Title: Lightweight Protection for Privacy in Offloaded Speech Understanding
----------------------------------------------------------------
